# Isentropic streamfunction

Computes the isentropic streamfunction $\Psi(z, \theta_e)$ (Pauluis and Mrowiec 2013) from 3D model output:

1. compute $\theta_e$ at every grid point with `isentropic.thermo`,
2. bin the vertical mass flux $\rho w$ in $(z, \theta_e)$ with `isentropic.iso_multi_integral`, level by level,
3. keep the conditional sums for each experiment (file) and time,
4. $\Psi(z, \theta_e)$ = cumulative sum over $\theta_e$ of the mass flux, divided by the domain area.

Use the **Python (iso)** kernel.

## Configuration
Edit the variable names to match the files (see the `ncdump -h` output).

In [ ]:
from pathlib import Path

DATA_DIR = Path("/scratch/ttg2015/mailbox/OPaul/2026-10-23_vddgrid_3d_30Days_Perpetual")
PATTERN = "*.nc"

# SAM 6.11 3D snapshot output
#   W (m/s, already at scalar levels), TABS (K), QV, QN (g/kg), p(z) (mb), PP (Pa)
#   QN = cloud liquid + ice, partitioned with SAM's temperature ramp between 253.16 and 273.16 K
VAR = dict(w="W", T="TABS", p="p", pp="PP", qv="QV", qn="QN")
Z_DIM = "z"
TIME_DIM = "time"
T_ICE, T_LIQ = 253.16, 273.16   # SAM's partition of QN into ice and liquid

# theta_e bins (K)
TH_MIN, TH_MAX, NBIN = 300.0, 370.0, 141

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import isentropic as iso
from isentropic import thermo

files = sorted(DATA_DIR.glob(PATTERN))
print(f"{len(files)} files")
ds0 = xr.open_dataset(files[0])
ds0

## Thermodynamic fields at one time

In [ ]:
def load_time(ds, it):
    """Return rho*w and theta_e (nz, ny, nx) and the density profile (nz) at time index it."""
    snap = ds.isel({TIME_DIM: it})
    get = lambda name: snap[name].values.astype(float)
    T = get(VAR["T"])
    qv = np.maximum(get(VAR["qv"]) * 1e-3, 1e-10)           # g/kg -> kg/kg
    qn = np.maximum(get(VAR["qn"]) * 1e-3, 0.0)
    liq_frac = np.clip((T - T_ICE) / (T_LIQ - T_ICE), 0.0, 1.0)
    ql, qi = qn * liq_frac, qn * (1.0 - liq_frac)
    p = get(VAR["p"])[:, None, None] * 100.0 + get(VAR["pp"])   # mb -> Pa, plus perturbation
    the = thermo.equivalent_potential_temperature(p, T, qv, ql, qi)
    rho = thermo.density(p, T, qv, ql, qi).mean((1, 2))        # density profile (anelastic)
    return rho[:, None, None] * get(VAR["w"]), the, rho


rhow, the, rho = load_time(ds0, 0)
print("theta_e range:", np.nanmin(the), np.nanmax(the))
print("rho*w range:", np.nanmin(rhow), np.nanmax(rhow))

## Conditional sums for each experiment and time
Each file is one experiment; its ID is the file name. The result is a dataset with dimensions
`(experiment, time, z, theta_e)`. Experiments with fewer times are padded with NaN.

In [ ]:
z = ds0[Z_DIM].values
dth = (TH_MAX - TH_MIN) / (NBIN - 1)
theta = TH_MIN + dth * np.arange(NBIN)


def process_experiment(f):
    """Conditional sums at every time of one experiment file."""
    with xr.open_dataset(f) as ds:
        nt = ds.sizes[TIME_DIM]
        MF, AREA, ATHE = (np.zeros((nt, len(z), NBIN)) for _ in range(3))
        RHO = np.zeros((nt, len(z)))
        for it in range(nt):
            rhow, the, RHO[it] = load_time(ds, it)
            mask = ~np.isfinite(the)
            out, AREA[it] = iso.iso_multi_integral(np.stack([rhow, the]), the, mask, 1.0,
                                                   th_min=TH_MIN, th_max=TH_MAX, nbin=NBIN)
            MF[it], ATHE[it] = out
            if it % 24 == 0:
                print(f.stem, "time", it, "/", nt)
        dims = ("time", "z", "theta_e")
        return xr.Dataset(
            {"MF": (dims, MF), "AREA": (dims, AREA), "ATHE": (dims, ATHE), "RHO": (("time", "z"), RHO)},
            coords={"time": ds[TIME_DIM].values, "z": z, "theta_e": theta},
        ).expand_dims(experiment=[f.stem])


sums = xr.concat([process_experiment(f) for f in files], dim="experiment", join="outer")
sums

## Streamfunction
$\Psi$ is computed separately for every experiment and time, at the upper edge of each $\theta_e$ bin.
The time-mean streamfunction uses the time-summed conditional sums.

In [ ]:
psi = iso.streamfunction(sums.MF.values, sums.AREA.values)          # (experiment, time, z, theta_e)
the_mean = sums.ATHE.sum("theta_e") / sums.AREA.sum("theta_e")         # (experiment, time, z)
ts = iso.two_stream(sums.MF.values, sums.AREA.values)                 # M_up, M_down: (experiment, time, z)

dims4 = ("experiment", "time", "z", "theta_e")
dims3 = ("experiment", "time", "z")
result = sums.assign(
    PSI=(dims4, psi),
    THE_MEAN=(dims3, the_mean.values),
    M_UP=(dims3, ts["M_up"]),
    M_DOWN=(dims3, ts["M_down"]),
)
result["MF"].attrs.update(long_name="conditional sum of rho*w (per grid point)", units="kg m-2 s-1")
result["AREA"].attrs.update(long_name="number of grid points in each bin")
result["PSI"].attrs.update(long_name="isentropic streamfunction", units="kg m-2 s-1",
                           note="evaluated at theta_e + dth/2")
result["RHO"].attrs.update(long_name="horizontal-mean density", units="kg m-3")
result.attrs.update(source=str(DATA_DIR), dth=dth)
result

In [ ]:
# Time-mean streamfunction for each experiment
mf_t = sums.MF.sum("time", skipna=True).values
area_t = sums.AREA.sum("time", skipna=True).values
psi_t = iso.streamfunction(mf_t, area_t)
the_t = sums.ATHE.sum("time", skipna=True).sum("theta_e").values / area_t.sum(-1)

nexp = len(sums.experiment)
fig, axes = plt.subplots(1, nexp, figsize=(5 * nexp, 5), squeeze=False, sharey=True)
lim = np.nanmax(np.abs(psi_t))
for e, ax in enumerate(axes[0]):
    pc = ax.pcolormesh(theta + dth / 2, z / 1e3, psi_t[e], cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
    ax.contour(theta + dth / 2, z / 1e3, psi_t[e], levels=np.linspace(-lim, lim, 20), colors="k", linewidths=0.4)
    ax.plot(the_t[e], z / 1e3, "k", lw=2)
    ax.set_title(str(sums.experiment.values[e]), fontsize=9)
    ax.set_xlabel(r"$\theta_e$ (K)")
axes[0, 0].set_ylabel("z (km)")
fig.colorbar(pc, ax=axes[0], label=r"time-mean $\Psi$ (kg m$^{-2}$ s$^{-1}$)")
plt.show()

## Two-stream mass fluxes (time mean)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for e, name in enumerate(sums.experiment.values):
    up = result.M_UP.isel(experiment=e).mean("time", skipna=True)
    line, = ax.plot(up, z / 1e3, label=str(name))
    ax.plot(-result.M_DOWN.isel(experiment=e).mean("time", skipna=True), z / 1e3, "--", color=line.get_color())
ax.set_xlabel(r"$M^+$ (solid), $-M^-$ (dashed) (kg m$^{-2}$ s$^{-1}$)")
ax.set_ylabel("z (km)")
ax.legend(fontsize=8)
plt.show()

## Save

In [ ]:
result.to_netcdf("iso_streamfunction.nc")